In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("/home/geraldine/Documents/Research/Projects/BayesGPT/")
print(sys.path)

['/home/a_huangm13/Documents/Research/Projects/BayesGPT', '/home/a_huangm13/Documents/Research/Projects/BayesGPT/bayesgpt', '/home/a_huangm13/Documents/Research/Projects/BayesGPT/tests', '/home/a_huangm13/.local/share/JetBrains/Toolbox/apps/pycharm/plugins/python-ce/helpers/pydev', '/home/a_huangm13/.local/share/JetBrains/Toolbox/apps/pycharm/plugins/python-ce/helpers/jupyter_debug', '/home/a_huangm13/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python311.zip', '/home/a_huangm13/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11', '/home/a_huangm13/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/lib-dynload', '', '/home/a_huangm13/Documents/Research/Projects/BayesGPT/.venv/lib/python3.11/site-packages', '/home/a_huangm13/Documents/Research/Projects/BayesGPT/.venv/lib/python3.11/site-packages/setuptools/_vendor', '/home/geraldine/Documents/Research/Projects/BayesGPT/']


In [3]:
import numpy as np

In [4]:
from bayesgpt.simulators.context_manager import ContextManager
from bayesgpt.simulators.model_family import NestedModelFamily
from bayesgpt.simulators.benchmarks.ddms.ddm import DDM
from bayesgpt.simulators.benchmarks.ddms.ddm_priors import ddm_baseline_priors

In [5]:
context_manager = ContextManager()

In [6]:
intrinsic_params = ["v", "a", "tau", "s_v", "s_tau"]

In [7]:
design_config = context_manager.build_design_config(
    intrinsic_params=intrinsic_params,
    regressed_params=["v", "a"],
    num_regressors=4,
    keep_intercept=True,
    add_interaction=True
)

In [8]:
for k, v in design_config.items():
    print(k, v)

1 ['v', 'a', 'tau', 's_v', 's_tau']
u_1 ['v', 'a']
u_2 ['v', 'a']
u_3 ['v', 'a']
u_4 ['v', 'a']
u_1:u_2 ['v', 'a']
u_1:u_3 ['v', 'a']
u_1:u_4 ['v', 'a']
u_2:u_3 ['v', 'a']
u_2:u_4 ['v', 'a']
u_3:u_4 ['v', 'a']


In [9]:
random_config = context_manager.build_random_design_config(
    intrinsic_params=intrinsic_params,
    num_regressors=2,
    free_intrinsics=intrinsic_params,
    fixed_intrinsics=[],
    keep_intercept=True,
    free_prob=0.5,
    add_interaction=True
)

In [10]:
for k, v in random_config.items():
    print(k, v)

1 ['v', 'a', 'tau', 's_v', 's_tau']
u_1 ['tau']
u_2 ['a', 'tau', 's_v']
u_1:u_2 ['tau']


In [11]:
X = context_manager.build_design_matrix(random_config, num_obs=10, keep_intercept=True, max_num_categories=3)

In [12]:
X.shape

(10, 7)

In [13]:
X

array([[1.        , 1.        , 0.        , 0.8996214 , 0.        ,
        0.8996214 , 0.        ],
       [1.        , 0.        , 0.        , 0.39352929, 0.        ,
        0.        , 0.        ],
       [1.        , 0.        , 1.        , 0.32912533, 0.        ,
        0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.98124574, 0.        ,
        0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.6655142 , 0.        ,
        0.        , 0.        ],
       [1.        , 0.        , 1.        , 0.15763323, 0.        ,
        0.        , 0.        ],
       [1.        , 0.        , 1.        , 0.22388428, 0.        ,
        0.        , 0.        ],
       [1.        , 0.        , 1.        , 0.9992991 , 0.        ,
        0.        , 0.        ],
       [1.        , 1.        , 0.        , 0.02695022, 0.        ,
        0.02695022, 0.        ],
       [1.        , 0.        , 0.        , 0.75368511, 0.        ,
        0.        , 0. 

In [14]:
param_mask = context_manager.build_parameter_mask(
    intrinsic_params=intrinsic_params,
    design_config=random_config,
    max_num_categories=3,
    keep_intercept=True,
)

In [15]:
param_mask.shape

(7, 5)

In [16]:
intrinsic_priors = context_manager.build_intrinsic_priors(
    prior_fun=ddm_baseline_priors(),
    free_intrinsics=intrinsic_params,
    fixed_intrinsics=[],
    fixed_values={}
)

In [17]:
intrinsic_priors

{'v': {'intercept': <function bayesgpt.simulators.benchmarks.ddms.ddm_priors.ddm_baseline_priors.<locals>.<lambda>()>,
  'slope': <function bayesgpt.simulators.context_manager.ContextManager.build_intrinsic_priors.<locals>.<lambda>(key='v')>},
 'a': {'intercept': <function bayesgpt.simulators.benchmarks.ddms.ddm_priors.ddm_baseline_priors.<locals>.<lambda>()>,
  'slope': <function bayesgpt.simulators.context_manager.ContextManager.build_intrinsic_priors.<locals>.<lambda>(key='a')>},
 'tau': {'intercept': <function bayesgpt.simulators.benchmarks.ddms.ddm_priors.ddm_baseline_priors.<locals>.<lambda>()>,
  'slope': <function bayesgpt.simulators.context_manager.ContextManager.build_intrinsic_priors.<locals>.<lambda>(key='tau')>},
 's_v': {'intercept': <function bayesgpt.simulators.benchmarks.ddms.ddm_priors.ddm_baseline_priors.<locals>.<lambda>()>,
  'slope': <function bayesgpt.simulators.context_manager.ContextManager.build_intrinsic_priors.<locals>.<lambda>(key='s_v')>},
 's_tau': {'inte

In [18]:
param_matrix = context_manager.sample_parameter_matrix(
    intrinsic_params=intrinsic_params,
    prior_fun=intrinsic_priors,
    parameter_mask=param_mask,
    keep_intercept=True,
)

In [19]:
param_matrix.shape

(7, 5)

In [20]:
ddm_family = NestedModelFamily(
    name="DDM",
    model=DDM(),
    prior_fun=ddm_baseline_priors(),
    mask_randomizer_kwargs=dict(
        free_intrinsics=["v", "a", "tau"],
        fixed_intrinsics=["s_v", "s_tau"],
        fixed_values={"s_v": 0, "s_tau": 0},
    )
)

In [21]:
sample_kwargs = {
    'num_regressors': 3,
    "max_num_categories": 3,
    "add_interaction": True,
}

samples = ddm_family.sample(
    num_obs=200,
    flatten_param_outputs=False,
    **sample_kwargs
)

In [22]:
for k, v in samples.items():
    if isinstance(v, np.ndarray):
        print(k, v.shape)
    elif isinstance(v, list):
        print(k, [v[i] for i in range(len(v))])
    elif isinstance(v, dict):
        print(k, v.keys())
    else:
        print(k, v)

model_name DDM
design_config dict_keys(['1', 'u_1', 'u_2', 'u_3', 'u_1:u_2', 'u_1:u_3', 'u_2:u_3'])
design_matrix (200, 13)
param_mask (13, 5)
param_matrix (13, 5)
sim_trials dict_keys(['rts', 'choices'])
discrete_mask (6,)
regressor_mask (13,)
max_num_regressors 5
keep_intercept True


In [23]:
samples["discrete_mask"]

array([ 0.,  1.,  0., -1., -1., -1.], dtype=float32)

In [24]:
samples["regressor_mask"]

array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.], dtype=float32)

In [34]:
ddm_samples = ddm_family.batch_sample(
    batch_size=1000,
    num_obs=500,
    max_num_regressors=3,
    max_num_categories=3,
    flatten_param_outputs=False,
    add_interaction=True
)

(500, 7) 7
(500, 13) 13
(500, 1) 1
(500, 7) 7
(500, 7) 7
(500, 5) 5
(500, 11) 11
(500, 1) 1
(500, 9) 9
(500, 9) 9
(500, 3) 3
(500, 1) 1
(500, 7) 7
(500, 5) 5
(500, 1) 1
(500, 3) 3
(500, 1) 1
(500, 5) 5
(500, 7) 7
(500, 3) 3
(500, 1) 1
(500, 9) 9
(500, 13) 13
(500, 1) 1
(500, 9) 9
(500, 3) 3
(500, 13) 13
(500, 13) 13
(500, 5) 5
(500, 13) 13
(500, 3) 3
(500, 3) 3
(500, 7) 7
(500, 3) 3
(500, 7) 7
(500, 1) 1
(500, 13) 13
(500, 5) 5
(500, 1) 1
(500, 3) 3
(500, 1) 1
(500, 1) 1
(500, 9) 9
(500, 5) 5
(500, 3) 3
(500, 5) 5
(500, 1) 1
(500, 9) 9
(500, 5) 5
(500, 7) 7
(500, 3) 3
(500, 9) 9
(500, 3) 3
(500, 1) 1
(500, 7) 7
(500, 9) 9
(500, 3) 3
(500, 1) 1
(500, 3) 3
(500, 1) 1
(500, 11) 11
(500, 7) 7
(500, 3) 3
(500, 3) 3
(500, 7) 7
(500, 3) 3
(500, 3) 3
(500, 3) 3
(500, 9) 9
(500, 3) 3
(500, 1) 1
(500, 5) 5
(500, 1) 1
(500, 3) 3
(500, 7) 7
(500, 13) 13
(500, 9) 9
(500, 1) 1
(500, 13) 13
(500, 1) 1
(500, 3) 3
(500, 11) 11
(500, 3) 3
(500, 3) 3
(500, 7) 7
(500, 9) 9
(500, 7) 7
(500, 3) 3
(500, 13) 